In [ ]:
pip install -q tensorflow-hub

In [ ]:
pip install -q tfds-nightly

In [ ]:
import tensorflow_hub as hub
import tensorflow_datasets as tfds

In [ ]:
import tensorflow as tf

In [ ]:

print("Version: ", tf.__version__)
print("Eager mode: ", tf.executing_eagerly())
print("Hub version: ", hub.__version__)
print("GPU is", "available" if tf.config.experimental.list_physical_devices("GPU") else "NOT AVAILABLE")

#1. loading and preping data

In [ ]:
train_data, validation_data, test_data = tfds.load(
    name="imdb_reviews",
    split=('train[:60%]', 'train[60%:]', 'test'),
    as_supervised=True
)

In [ ]:

print())

In [ ]:
#Exploring the data printing the first 10 examples
train_examples_batch, train_labels_batch = next(iter(train_data.batch(10)))

In [ ]:
train_examples_batch

In [ ]:
train_labels_batch

#2.Create a model

#Using transfer learning

In [19]:
#Downloading the weights
embedding = "https://tfhub.dev/google/tf2-preview/gnews-swivel-20dim/1"
hub_layer = hub.KerasLayer(embedding, input_shape=[], 
                           dtype=tf.string, trainable=True)
hub_layer(train_examples_batch[:3])


<tf.Tensor: shape=(3, 20), dtype=float32, numpy=
array([[ 1.765786  , -3.882232  ,  3.9134233 , -1.5557289 , -3.3362343 ,
        -1.7357955 , -1.9954445 ,  1.2989551 ,  5.081598  , -1.1041286 ,
        -2.0503852 , -0.72675157, -0.65675956,  0.24436149, -3.7208383 ,
         2.0954835 ,  2.2969332 , -2.0689783 , -2.9489717 , -1.1315987 ],
       [ 1.8804485 , -2.5852382 ,  3.4066997 ,  1.0982676 , -4.056685  ,
        -4.891284  , -2.785554  ,  1.3874227 ,  3.8476458 , -0.9256538 ,
        -1.896706  ,  1.2113281 ,  0.11474707,  0.76209456, -4.8791065 ,
         2.906149  ,  4.7087674 , -2.3652055 , -3.5015898 , -1.6390051 ],
       [ 0.71152234, -0.6353217 ,  1.7385626 , -1.1168286 , -0.5451594 ,
        -1.1808156 ,  0.09504455,  1.4653089 ,  0.66059524,  0.79308075,
        -2.2268345 ,  0.07446612, -1.4075904 , -0.70645386, -1.907037  ,
         1.4419787 ,  1.9551861 , -0.42660055, -2.8022065 ,  0.43727064]],
      dtype=float32)>

In [20]:
model = tf.keras.Sequential()
model.add(hub_layer)
model.add(tf.keras.layers.Dense(16, activation='relu'))
model.add(tf.keras.layers.Dense(1))

model.summary()


Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
keras_layer (KerasLayer)     (None, 20)                400020    
_________________________________________________________________
dense (Dense)                (None, 16)                336       
_________________________________________________________________
dense_1 (Dense)              (None, 1)                 17        
Total params: 400,373
Trainable params: 400,373
Non-trainable params: 0
_________________________________________________________________


In [21]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              metrics=['accuracy'])


In [22]:
history = model.fit(train_data.shuffle(10000).batch(512),
                    epochs=20,
                    validation_data=validation_data.batch(512),
                    verbose=1)


Epoch 1/20
30/30 [==============================] - 2s 82ms/step - loss: 0.9499 - accuracy: 0.4322 - val_loss: 0.7889 - val_accuracy: 0.4818
Epoch 2/20
30/30 [==============================] - 2s 80ms/step - loss: 0.7347 - accuracy: 0.5013 - val_loss: 0.6724 - val_accuracy: 0.5563
Epoch 3/20
30/30 [==============================] - 2s 79ms/step - loss: 0.6398 - accuracy: 0.5962 - val_loss: 0.6104 - val_accuracy: 0.6440
Epoch 4/20
30/30 [==============================] - 2s 80ms/step - loss: 0.5853 - accuracy: 0.6621 - val_loss: 0.5697 - val_accuracy: 0.6834
Epoch 5/20
30/30 [==============================] - 2s 80ms/step - loss: 0.5432 - accuracy: 0.7044 - val_loss: 0.5359 - val_accuracy: 0.7269
Epoch 6/20
30/30 [==============================] - 2s 79ms/step - loss: 0.5042 - accuracy: 0.7432 - val_loss: 0.5002 - val_accuracy: 0.7270
Epoch 7/20
30/30 [==============================] - 2s 81ms/step - loss: 0.4652 - accuracy: 0.7685 - val_loss: 0.4661 - val_accuracy: 0.7613
Epoch 8/20
30

In [23]:
results = model.evaluate(test_data.batch(512), verbose=2)

for name, value in zip(model.metrics_names, results):
  print("%s: %.3f" % (name, value))


49/49 - 2s - loss: 0.3220 - accuracy: 0.8573
loss: 0.322
accuracy: 0.857
